# Managing Big Data

In [1]:
# connect to Enterprise GIS
from arcgis.gis import GIS
import arcgis.geoanalytics

portal_gis = GIS("https://ndhwks6.esri.com/portal", "admin", 'esri.agp', verify_cert=False)

In [2]:
item = portal_gis.content.get('c81431e0194a42cc9e6f4336c76a70b6')
usa_counties_lyr = item.layers[0]

In [3]:
usa_counties_lyr

<FeatureLayer url:"https://ndhwks6.esri.com/server/rest/services/Hosted/usaCounties/FeatureServer/0">

In [4]:
bigdata_datastore_manager = arcgis.geoanalytics.get_datastores()
bigdata_datastore_manager

<DatastoreManager for https://ndhwks6.esri.com:6443/arcgis/admin>

In [5]:
data_item2 = bigdata_datastore_manager.add_bigdata("air_quality", r"\\NDHWKS6\Users\arcgis\Documents\air")

Big Data file share exists for air_quality


In [6]:
search_result1 = portal_gis.content.search("bigDataFileShares_air_quality", item_type = "big data file share")
search_result1

[<Item title:"bigDataFileShares_air_quality" type:Big Data File Share owner:admin>]

In [7]:
air_lyr = search_result1[0].layers[0]

In [8]:
air_lyr

<Layer url:"https://ndhwks6.esri.com/server/rest/services/DataStoreCatalogs/bigDataFileShares_air_quality/BigDataCatalogServer/air_quality">

In [9]:
search_result2 = portal_gis.content.search("bigDataFileShares_all_hurricanes", item_type = "big data file share")
search_result2

[<Item title:"bigDataFileShares_all_hurricanes" type:Big Data File Share owner:admin>]

In [10]:
hurr = search_result2[0].layers[0]

In [11]:
search_result3 = portal_gis.content.search("bigDataFileShares_ServiceCallsOrleans", item_type = "big data file share")[0]
search_result3

<Item title:"bigDataFileShares_ServiceCallsOrleans" type:Big Data File Share owner:admin>

In [12]:
calls = search_result3.layers[0]

In [13]:
from arcgis.features import FeatureLayer

In [14]:
blk_grp_lyr = FeatureLayer('https://services.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/USA_Census_BlockGroup_Areas_analysis_trim/FeatureServer/0')

In [15]:
blk_grp_lyr.filter = "County='Orleans'"

## Append Data

In [16]:
from arcgis.geoanalytics.manage_data import append_data

input_lyr = portal_gis.content.get('b39deed705144a2a90c5eaf5a44f5a14').layers[0]
append_lyr = portal_gis.content.get('78d4b1914bba4e09b9e8006fa6a3157c').layers[0]

append_data(input_layer=input_lyr, append_layer=append_lyr)

## Calculate Fields

In [17]:
from arcgis.geoanalytics.manage_data import calculate_fields

In [18]:
calc = calculate_fields(input_layer=hurr,
                 field_name="avg",
                 data_type="Double",
                 expression='max($feature["wind_wmo1"],$feature["pres_wmo1"])')

In [19]:
calc.delete()

True

## Clip Layer

In [20]:
from arcgis.geoanalytics.manage_data import clip_layer
from datetime import datetime as dt

In [21]:
clip_result = clip_layer(calls, blk_grp_lyr, output_name="service calls in new Orleans" + str(dt.now().microsecond))

In [22]:
clip_result.delete()

True

## Copy To Datastore

In [23]:
from arcgis.geoanalytics.manage_data import copy_to_data_store

In [24]:
copy = copy_to_data_store(hurr)

In [25]:
copy.delete()

True

## Dissolve Boundaries

In [26]:
from arcgis.geoanalytics.manage_data import dissolve_boundaries

In [27]:
diss = dissolve_boundaries(input_layer=blk_grp_lyr, 
                    dissolve_fields='County', 
                    output_name='dissolved by countyfp'+ str(dt.now().microsecond))

<Item title:"dissolved_by_countyfp" type:Feature Layer Collection owner:admin>

In [28]:
diss.delete()

NameError: name 'diss' is not defined

## Merge Layers

In [29]:
from arcgis.geoanalytics.manage_data import merge_layers

In [30]:
bigdata_datastore_manager.add_bigdata("nyc1", r"\\NDHWKS6\Users\arcgis\Documents\NYC_taxi_data1")
bigdata_datastore_manager.add_bigdata("nyc2", r"\\NDHWKS6\Users\arcgis\Documents\NYC_taxi_data2")

Big Data file share exists for nyc1
Big Data file share exists for nyc2


<Datastore title:"/bigDataFileShares/nyc2" type:"bigDataFileShare">

In [31]:
taxi_search = portal_gis.content.search("bigDataFileShares_nyc", item_type = "big data file share")
taxi_search

[<Item title:"bigDataFileShares_nyc1" type:Big Data File Share owner:admin>,
 <Item title:"bigDataFileShares_nyc2" type:Big Data File Share owner:admin>]

In [32]:
taxi1 = taxi_search[0].layers[0]
taxi2 = taxi_search[1].layers[0]

In [33]:
merge = merge_layers(taxi1, taxi2, output_name='merged layers'+ str(dt.now().microsecond))

In [34]:
merge.delete()

True

## Overlay Data

In [35]:
from arcgis.geoanalytics.manage_data import overlay_data

In [36]:
intersect = overlay_data(calls, blk_grp_lyr, output_name='intersected features'+ str(dt.now().microsecond))

<Item title:"intersected_features410836" type:Feature Layer Collection owner:admin>

In [ ]:
intersect.delete()

## Run Python Script

In [37]:
from arcgis.geoanalytics.manage_data import run_python_script

In [40]:
def average():
    from datetime import datetime as dt
    spark_df = layers[0] #
    spark_df = spark_df.filter(spark_df['Parameter Name'] == 'PM2.5 - Local Conditions') #pyspark filter
    res = geoanalytics.join_features(target_layer=layers[1],  
                                     join_layer=spark_df, 
                                     join_operation="JoinOneToOne",
                                     summary_fields=[{'statisticType' : 'mean', 'onStatisticField' : 'Sample Measurement'}],
                                     spatial_relationship='Contains')
    res.write.format("webgis").save("average_pm_by_county"+ str(dt.now().microsecond))

In [ ]:
run_python_script(average, [air_lyr, usa_counties_lyr])